# Compare framing detection across two models

This standalone notebook compares the Gemma and Qwen outputs, displays the statistics and charts, and exports segment-level disagreements as CSV files. It contains the comparison code, so the Python script is not required.

**How to use:** Select a Python 3.10+ kernel, edit the file paths and options in the settings cell, then choose **Run All**. All results are written to `OUTPUT_DIR`, relative to the notebook's working directory unless you enter an absolute path. Use a fresh output directory when changing export options.

The notebook is supplied without precomputed results: access restrictions prevented a full run against the original Desktop files. The comparison and export workflow was checked on test data.

If needed, run this in a new code cell once, then restart the kernel:

```python
%pip install pandas numpy matplotlib
```

To install Jupyter itself from a terminal: `python3 -m pip install jupyterlab`, then launch `jupyter lab`.

## 1. Settings

The original dataset paths are filled in below. Model A is Gemma; model B is Qwen. Leave `JOIN_KEYS = None` to match on `_row_id`, falling back to `doc_id` and `segment_index`. Group summaries use model A's metadata.

In [ ]:
from pathlib import Path

CSV_A = Path('data/frames_debate_complete_gemma3_27b.csv')
CSV_B = Path('data/frames_debate_complete_qwen3.5_35b.csv')
OUTPUT_DIR = Path('output')

NAME_A = 'Gemma 3 27B'
NAME_B = 'Qwen 3.5 35B'
JOIN_KEYS = None  # Example: ['doc_id', 'segment_index']
GROUP_BY = ['year', 'source']  # Use [] to disable grouped summaries.
EXPORT_DISAGREEMENTS = True
MAKE_PLOTS = True
PREVIEW_ROWS = 10

## 2. Comparison definitions

- **Frame label:** equality of `NOT_FRAMED`, `SINGLE_FRAME`, or `MULTIPLE_FRAMES`.
- **Types without polarity:** exact equality of unordered sets of base types.
- **Types with polarity:** exact equality of unordered sets of type/polarity pairs.
- **Joint comparisons:** both the label and the corresponding type set must agree.

Order and duplicate frame tokens do not matter. Case and whitespace are normalized. Two empty type sets are valid agreement when both labels are `NOT_FRAMED`. Processing failures and segment-text mismatches are excluded. Missing or invalid types are excluded from type statistics; usable labels can still contribute to label agreement. Every metric shows its own eligible denominator.

Cohen's kappa corrects for agreement expected from the marginal distributions. For type sets, each complete set is treated as a nominal category. Undefined statistics are blank in exported CSVs. These measures describe agreement rather than accuracy against human annotations.

In [ ]:
from __future__ import annotations

import json
import re
import numpy as np
import pandas as pd

LABELS = {'NOT_FRAMED', 'SINGLE_FRAME', 'MULTIPLE_FRAMES'}
NAMES = {
    'frame_label': 'Frame label',
    'frame_types_without_polarity': 'Frame types without polarity',
    'frame_types_with_polarity': 'Frame types with polarity',
    'label_and_types_without_polarity': 'Label + types without polarity',
    'label_and_types_with_polarity': 'Label + types with polarity',
}

### Parsing, alignment, and quality checks
These helpers preserve identifiers and Chinese text, validate joins, and distinguish empty frame sets from failed predictions.

In [ ]:
def normalize(value: str) -> str:
    return " ".join(value.strip().split()).casefold()



def parse_frames(value: str) -> tuple[frozenset[str], frozenset[str]]:
    """Order and duplicate tokens do not matter; polarity must be parenthesized."""
    signed, bases = set(), set()
    if not value.strip():
        return frozenset(), frozenset()
    for token in value.split("|"):
        match = re.fullmatch(r"\s*([^()|]+?)\s*\(\s*([^()|]+?)\s*\)\s*", token)
        if not match:
            raise ValueError(f"Invalid frame token: {token!r}; expected 'type (polarity)'")
        base, polarity = (normalize(x) for x in match.groups())
        signed.add(f"{base} ({polarity})")
        bases.add(base)
    return frozenset(signed), frozenset(bases)



def format_frame_set(values) -> str:
    return "|".join(sorted(values))



def jaccard(a, b) -> float:
    return len(a & b) / len(a | b) if a | b else 1.0



def kappa(a, b) -> float:
    """Unweighted Cohen's kappa; NaN when empty or expected agreement is 1."""
    if not len(a):
        return float("nan")
    a, b = pd.Series(list(a)), pd.Series(list(b))
    observed = float((a == b).mean())
    expected = float(a.value_counts(normalize=True).mul(
        b.value_counts(normalize=True), fill_value=0).sum())
    return (observed - expected) / (1 - expected) if expected < 1 - 1e-12 else float("nan")



def write_csv(frame, path, index=False):
    frame.to_csv(path, index=index, encoding="utf-8-sig")



def load(path: Path) -> pd.DataFrame:
    # Keep literal NA strings and identifiers intact; only blank fields are blank.
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    missing = {"frame_label", "frame_type"} - set(df.columns)
    if missing:
        raise ValueError(f"{path}: missing columns {sorted(missing)}")
    if "frame_processing_error" not in df:
        df["frame_processing_error"] = ""
    return df



def align(a, b, keys):
    for name, df in [("A", a), ("B", b)]:
        missing = set(keys) - set(df.columns)
        if missing:
            raise ValueError(f"Model {name}: missing join columns {sorted(missing)}")
        if df[keys].apply(lambda s: s.str.strip().eq("")).any().any():
            raise ValueError(f"Model {name}: blank join identifiers")
        if df.duplicated(keys).any():
            examples = df.loc[df.duplicated(keys, keep=False), keys].head().to_dict("records")
            raise ValueError(f"Model {name}: duplicate join identifiers: {examples}")
    joined = a.merge(b, on=keys, how="outer", suffixes=("_a", "_b"),
                     validate="one_to_one", indicator=True)
    joined["_merge"] = joined["_merge"].astype(str)
    return joined.fillna("")



def prepare(matched):
    """Keep all matched rows; attach quality reasons and metric eligibility."""
    result = matched.copy()
    records = []
    for _, row in result.iterrows():
        reasons = []
        text_mismatch = ("segment_text_a" in row and "segment_text_b" in row
                         and row.segment_text_a != row.segment_text_b)
        if text_mismatch:
            reasons.append("segment_text_mismatch")
        common_ok = not text_mismatch
        parsed = {}
        label_ok, type_ok = {}, {}
        for side in ("a", "b"):
            label = row[f"frame_label_{side}"].strip().upper()
            error = normalize(row.get(f"frame_processing_error_{side}", ""))
            if error in {"true", "1", "yes"}:
                reasons.append(f"processing_error_{side}")
                common_ok = False
            elif error not in {"", "false", "0", "no"}:
                reasons.append(f"unknown_processing_error_flag_{side}")
                common_ok = False
            label_ok[side] = label in LABELS
            if not label_ok[side]:
                reasons.append(f"missing_or_unknown_frame_label_{side}")
            try:
                signed, bases = parse_frames(row[f"frame_type_{side}"])
                # Blank NOT_FRAMED is a valid empty set, never a missing prediction.
                type_ok[side] = label_ok[side] and (
                    (label == "NOT_FRAMED" and not signed)
                    or (label in {"SINGLE_FRAME", "MULTIPLE_FRAMES"} and bool(signed)))
                if not type_ok[side]:
                    reasons.append(f"missing_or_inconsistent_frame_type_{side}")
                if label == "SINGLE_FRAME" and len(bases) != 1 and signed:
                    reasons.append(f"label_type_count_warning_{side}")
                if label == "MULTIPLE_FRAMES" and len(bases) < 2 and signed:
                    reasons.append(f"label_type_count_warning_{side}")
            except ValueError:
                signed, bases = frozenset(), frozenset()
                type_ok[side] = False
                reasons.append(f"malformed_frame_type_{side}")
            parsed[side] = (signed, bases, label)
        label_valid = common_ok and all(label_ok.values())
        types_valid = common_ok and all(type_ok.values())
        sa, ba, la = parsed["a"]
        sb, bb, lb = parsed["b"]
        record = {"quality_reasons": ";".join(reasons),
                  "frame_label_eligible": label_valid,
                  "frame_types_without_polarity_eligible": types_valid,
                  "frame_types_with_polarity_eligible": types_valid,
                  "label_and_types_without_polarity_eligible": label_valid and types_valid,
                  "label_and_types_with_polarity_eligible": label_valid and types_valid,
                  "frame_label_agree": la == lb if label_valid else pd.NA,
                  "frame_types_without_polarity_agree": ba == bb if types_valid else pd.NA,
                  "frame_types_with_polarity_agree": sa == sb if types_valid else pd.NA,
                  "label_and_types_without_polarity_agree": la == lb and ba == bb if label_valid and types_valid else pd.NA,
                  "label_and_types_with_polarity_agree": la == lb and sa == sb if label_valid and types_valid else pd.NA,
                  "jaccard_without_polarity": jaccard(ba, bb) if types_valid else np.nan,
                  "jaccard_with_polarity": jaccard(sa, sb) if types_valid else np.nan,
                  "polarity_only_disagreement": types_valid and ba == bb and sa != sb}
        for side, (signed, bases, label) in parsed.items():
            record.update({f"signed_set_{side}": signed, f"base_set_{side}": bases,
                           f"normalized_label_{side}": label,
                           f"normalized_types_with_polarity_{side}": format_frame_set(signed),
                           f"normalized_types_without_polarity_{side}": format_frame_set(bases)})
        for mode, x, y in [("with_polarity", sa, sb), ("without_polarity", ba, bb)]:
            record[f"types_only_a_{mode}"] = format_frame_set(x - y) if types_valid else ""
            record[f"types_only_b_{mode}"] = format_frame_set(y - x) if types_valid else ""
        records.append(record)
    # Fixed empty schema also supports files with no overlapping keys.
    if not records:
        empty = prepare(pd.DataFrame([{
            "frame_label_a": "", "frame_label_b": "", "frame_type_a": "", "frame_type_b": ""
        }])).iloc[:0]
        return pd.concat([result.reset_index(drop=True), empty.drop(columns=[
            "frame_label_a", "frame_label_b", "frame_type_a", "frame_type_b"])], axis=1)
    return pd.concat([result.reset_index(drop=True), pd.DataFrame(records)], axis=1)



def exportable(df):
    return df.drop(columns=["signed_set_a", "signed_set_b", "base_set_a", "base_set_b"], errors="ignore")

### Statistics and chart helpers
Per-frame agreement includes joint absences. Positive agreement omits joint absences: `2 × both_present / (2 × both_present + a_only + b_only)`.

In [ ]:
def summarize(df):
    rows = []
    for metric in NAMES:
        valid = df.loc[df[f"{metric}_eligible"].eq(True)]
        count = int(valid[f"{metric}_agree"].eq(True).sum())
        if metric == "frame_label":
            x, y = valid.normalized_label_a, valid.normalized_label_b
        elif metric.startswith("label_and"):
            mode = "with_polarity" if metric.endswith("_with_polarity") else "without_polarity"
            x = list(zip(valid.normalized_label_a, valid[f"normalized_types_{mode}_a"]))
            y = list(zip(valid.normalized_label_b, valid[f"normalized_types_{mode}_b"]))
            # JSON encodes tuples unambiguously for categorical kappa.
            x, y = [json.dumps(v) for v in x], [json.dumps(v) for v in y]
        else:
            mode = metric.removeprefix("frame_types_")
            x, y = valid[f"normalized_types_{mode}_a"], valid[f"normalized_types_{mode}_b"]
        rows.append({"metric": metric, "n_eligible": len(valid), "n_agree": count,
                     "n_disagree": len(valid) - count, "n_excluded": len(df) - len(valid),
                     "agreement_pct": 100 * count / len(valid) if len(valid) else np.nan,
                     "cohens_kappa": kappa(x, y)})
    return pd.DataFrame(rows)



def per_frame_statistics(df):
    valid = df.loc[df.frame_types_with_polarity_eligible.eq(True)]
    rows = []
    for mode, column in [("without_polarity", "base_set"), ("with_polarity", "signed_set")]:
        vocabulary = sorted(set().union(*valid[f"{column}_a"], *valid[f"{column}_b"]))
        for frame in vocabulary:
            a = valid[f"{column}_a"].map(lambda s: frame in s)
            b = valid[f"{column}_b"].map(lambda s: frame in s)
            both = int((a & b).sum())
            a_only, b_only = int((a & ~b).sum()), int((~a & b).sum())
            neither = int((~a & ~b).sum())
            shared_polarity = []
            if mode == "without_polarity":
                for sa, sb in zip(valid.signed_set_a, valid.signed_set_b):
                    pa = {s for s in sa if s.rsplit(" (", 1)[0] == frame}
                    pb = {s for s in sb if s.rsplit(" (", 1)[0] == frame}
                    if pa and pb:
                        shared_polarity.append(pa == pb)
            rows.append({"mode": mode, "frame": frame, "n_eligible": len(valid),
                         "a_present": int(a.sum()), "b_present": int(b.sum()),
                         "a_prevalence_pct": 100 * a.mean(), "b_prevalence_pct": 100 * b.mean(),
                         "both_present": both, "a_only": a_only, "b_only": b_only,
                         "neither_present": neither, "agreement_pct": 100 * (a == b).mean(),
                         "cohens_kappa": kappa(a, b),
                         "positive_agreement_pct": 200 * both / (2 * both + a_only + b_only)
                         if 2 * both + a_only + b_only else np.nan,
                         "polarity_shared_frame_n": len(shared_polarity) if mode == "without_polarity" else np.nan,
                         "polarity_shared_frame_agreement_pct": 100 * np.mean(shared_polarity)
                         if shared_polarity else np.nan})
    return pd.DataFrame(rows, columns=["mode", "frame", "n_eligible", "a_present", "b_present",
        "a_prevalence_pct", "b_prevalence_pct", "both_present", "a_only", "b_only",
        "neither_present", "agreement_pct", "cohens_kappa", "positive_agreement_pct",
        "polarity_shared_frame_n", "polarity_shared_frame_agreement_pct"])



def charts(summary, confusion, per_frame, distribution, out, names):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
    def save(fig, name):
        fig.savefig(out / name, dpi=180, bbox_inches="tight")
        plt.close(fig)
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh([NAMES[m] for m in summary.metric], summary.agreement_pct, color="#286d96")
    ax.set_xlim(0, 115)
    ax.set_xlabel("Exact agreement (%)")
    ax.set_title(f"{names[0]} vs {names[1]}")
    for bar, (_, r) in zip(bars, summary.iterrows()):
        if pd.notna(r.agreement_pct):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                    f"{r.agreement_pct:.1f}% (n={r.n_eligible:,})", va="center", fontsize=9)
        else:
            ax.text(1, bar.get_y() + bar.get_height()/2,
                    "No eligible segments (n=0)", va="center", fontsize=9)
    ax.invert_yaxis()
    save(fig, "agreement.png")
    fig, ax = plt.subplots(figsize=(8, 6))
    matrix = confusion.to_numpy(dtype=int)
    img = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=max(1, matrix.max()))
    for i in range(len(confusion)):
        total = matrix[i].sum()
        for j in range(len(confusion.columns)):
            ax.text(j, i, f"{matrix[i,j]:,}\n{100*matrix[i,j]/total:.1f}%" if total else "0\nn/a",
                    ha="center", va="center", color="white" if matrix[i,j] > matrix.max()/2 else "black")
    ax.set_xticks(range(len(confusion.columns)), confusion.columns, rotation=20, ha="right")
    ax.set_yticks(range(len(confusion)), confusion.index)
    ax.set_xlabel(names[1]); ax.set_ylabel(names[0])
    ax.set_title("Frame label cross-tabulation: count and row %")
    fig.colorbar(img, ax=ax, label="Segments")
    save(fig, "frame_label_confusion.png")
    for mode in ["without_polarity", "with_polarity"]:
        data = per_frame.loc[per_frame["mode"].eq(mode)].sort_values("a_prevalence_pct")
        if data.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, max(4, len(data) * .4)))
        y = np.arange(len(data))
        ax.barh(y-.18, data.a_prevalence_pct, height=.35, label=names[0], color="#286d96")
        ax.barh(y+.18, data.b_prevalence_pct, height=.35, label=names[1], color="#df8b3e")
        ax.set_yticks(y, data.frame)
        ax.set_xlabel("Segments containing frame (%)")
        ax.set_xlim(0, 100); ax.legend()
        ax.set_title(f"Frame prevalence {mode.replace('_', ' ')} (n={int(data.n_eligible.iloc[0]):,})")
        save(fig, f"frame_prevalence_{mode}.png")
    fig, ax = plt.subplots(figsize=(8, 5))
    categories = sorted(LABELS)
    y = np.arange(len(categories))
    for i, side in enumerate(("a", "b")):
        values = distribution.loc[distribution.model_side.eq(side)].set_index("frame_label").reindex(categories)
        ax.bar(y + (i-.5)*.35, values.prevalence_pct, width=.35, label=names[i],
               color=["#286d96", "#df8b3e"][i])
    ax.set_xticks(y, categories); ax.set_ylabel("Eligible segments (%)")
    ax.set_title("Frame label distributions on the same paired population")
    ax.legend(); save(fig, "frame_label_distribution.png")

## 3. Load and align segments

Matching uses identifiers rather than row position. Blank or duplicate identifiers stop execution. Unmatched segments are exported separately and excluded from agreement percentages.

In [ ]:
a, b = load(CSV_A), load(CSV_B)
keys = JOIN_KEYS or (["_row_id"] if "_row_id" in a and "_row_id" in b else ["doc_id", "segment_index"])
def model_name(df, override, path):
    values = df.get("framing_model", pd.Series(dtype=str))
    unique = values[values.ne("")].unique()
    return override or (unique[0] if len(unique) == 1 else path.stem)
names = (model_name(a, NAME_A, CSV_A), model_name(b, NAME_B, CSV_B))
joined = align(a, b, keys)

display(pd.DataFrame({'model': names, 'input_rows': [len(a), len(b)]}))
display(joined['_merge'].value_counts().rename('segments').to_frame())

## 4. Quality checks and headline agreement

Review the eligible and excluded counts before interpreting percentages. Warnings about label/type counts are reported but do not exclude otherwise usable predictions. `quality_issues.csv` keeps the original model outputs for inspection.

In [ ]:
out = OUTPUT_DIR
out.mkdir(parents=True, exist_ok=True)
write_csv(joined.loc[joined._merge.ne("both")], out / "unmatched_segments.csv")
pairs = prepare(joined.loc[joined._merge.eq("both")])
write_csv(exportable(pairs), out / "segment_comparisons.csv")
write_csv(exportable(pairs.loc[pairs.quality_reasons.ne("")]), out / "quality_issues.csv")
summary = summarize(pairs)
write_csv(summary, out / "agreement_summary.csv")

display(summary.round({'agreement_pct': 2, 'cohens_kappa': 4}))
quality_rows = pairs.loc[pairs.quality_reasons.ne('')]
print(f'Matched pairs: {len(pairs):,}; pairs with quality issues or warnings: {len(quality_rows):,}')
display(exportable(quality_rows).head(PREVIEW_ROWS))

## 5. Partial overlap and conditional polarity agreement

Jaccard similarity measures `|intersection| / |union|`. Two empty sets have similarity 1. The notebook also reports Jaccard on pairs where either model detects a frame, making the effect of joint absences visible.

Conditional polarity agreement compares signed sets only when the models predict the same **nonempty** base-type set.

In [ ]:
types = pairs.loc[pairs.frame_types_with_polarity_eligible.eq(True)]
overlap_rows = []
for mode in ("without_polarity", "with_polarity"):
    # Restrict the secondary overlap measure to pairs where either model has a frame.
    nonempty = types.loc[(types.base_set_a.map(len) + types.base_set_b.map(len)).gt(0)]
    for population, df in [("all_eligible", types), ("either_model_framed", nonempty)]:
        overlap_rows.append({"mode": mode, "population": population, "n": len(df),
                             "mean_jaccard": df[f"jaccard_{mode}"].mean()})
same_nonempty = types.loc[types.frame_types_without_polarity_agree.eq(True) & types.base_set_a.map(len).gt(0)]
overlap_rows.append({"mode": "polarity_agreement_given_identical_nonempty_base_sets",
                     "population": "identical_nonempty_base_sets", "n": len(same_nonempty),
                     "agreement_pct": 100 * same_nonempty.frame_types_with_polarity_agree.mean()
                     if len(same_nonempty) else np.nan})
overlap = pd.DataFrame(overlap_rows)
write_csv(overlap, out / "overlap_summary.csv")

display(overlap.round(4))

## 6. Per-frame statistics and label distributions

Both models use the same eligible paired population. Frame prevalence can sum above 100% because a segment can contain multiple frames. Per-frame polarity agreement is calculated on segments where both models identify that particular base frame.

In [ ]:
frame_stats = per_frame_statistics(pairs)
write_csv(frame_stats, out / "per_frame_statistics.csv")
label_pairs = pairs.loc[pairs.frame_label_eligible.eq(True)]
confusion = pd.crosstab(label_pairs.normalized_label_a, label_pairs.normalized_label_b).reindex(
    index=sorted(LABELS), columns=sorted(LABELS), fill_value=0)
confusion.index.name = f"{names[0]} (rows) vs {names[1]} (columns)"
write_csv(confusion, out / "frame_label_confusion_counts.csv", index=True)
write_csv(confusion.div(confusion.sum(axis=1).replace(0, np.nan), axis=0) * 100,
          out / "frame_label_confusion_row_percent.csv", index=True)
distribution = []
for side, name in zip(("a", "b"), names):
    counts = label_pairs[f"normalized_label_{side}"].value_counts()
    for label in sorted(LABELS):
        count = int(counts.get(label, 0))
        distribution.append({"model_side": side, "model": name, "frame_label": label,
                             "count": count, "n_eligible": len(label_pairs),
                             "prevalence_pct": 100 * count / len(label_pairs) if len(label_pairs) else np.nan})
distribution = pd.DataFrame(distribution)
write_csv(distribution, out / "frame_label_distribution.csv")

display(frame_stats.round(3))
display(confusion)
display(distribution.round(2))

## 7. Agreement by year and source
Change `GROUP_BY` in settings to compare other metadata fields. Each group and metric retains its own denominator.

In [ ]:
grouped = []
group_columns = {}
for group in GROUP_BY:
    column = group if group in keys else f"{group}_a"
    if column not in pairs:
        if group in a:
            column = group
        else:
            print(f"Skipping absent grouping column: {group}")
            continue
    group_columns[group] = column
    for value, subset in pairs.groupby(column, dropna=False, sort=True):
        table = summarize(subset)
        table.insert(0, "group_value", value)
        table.insert(0, "group_column", group)
        grouped.append(table)
write_csv(pd.concat(grouped, ignore_index=True) if grouped else pd.DataFrame(
    columns=["group_column", "group_value", *summary.columns]), out / "agreement_by_group.csv")

group_summary = pd.concat(grouped, ignore_index=True) if grouped else pd.DataFrame()
display(group_summary.head(30).round(3))

## 8. Export and inspect disagreements

Separate CSVs contain label disagreements, type disagreements with and without polarity, joint disagreements, polarity-only disagreements, and their union. They retain both models' text, explanations, evidence, and raw predictions with `_a` and `_b` suffixes. Invalid or failed predictions appear in the quality export.

Set `EXPORT_DISAGREEMENTS = False` to skip the separate disagreement CSVs; `segment_comparisons.csv` is still generated.

In [ ]:
if EXPORT_DISAGREEMENTS:
    for metric in NAMES:
        mask = pairs[f"{metric}_eligible"].eq(True) & pairs[f"{metric}_agree"].eq(False)
        write_csv(exportable(pairs.loc[mask]), out / f"disagreements_{metric}.csv")
    write_csv(exportable(pairs.loc[pairs.polarity_only_disagreement.eq(True)]),
              out / "disagreements_polarity_only.csv")
    any_disagreement = pd.Series(False, index=pairs.index)
    for metric in list(NAMES)[:3]:
        any_disagreement |= pairs[f"{metric}_eligible"].eq(True) & pairs[f"{metric}_agree"].eq(False)
    write_csv(exportable(pairs.loc[any_disagreement]), out / "disagreements_any.csv")

disagreement_counts = []
for metric in NAMES:
    mask = pairs[f'{metric}_eligible'].eq(True) & pairs[f'{metric}_agree'].eq(False)
    disagreement_counts.append({'comparison': NAMES[metric], 'segments': int(mask.sum())})
display(pd.DataFrame(disagreement_counts))

In [ ]:
# Change this value to inspect another comparison from the table above.
INSPECT_METRIC = 'frame_types_with_polarity'

if INSPECT_METRIC not in NAMES:
    raise ValueError(f'Choose one of: {list(NAMES)}')
mask = pairs[f'{INSPECT_METRIC}_eligible'].eq(True) & pairs[f'{INSPECT_METRIC}_agree'].eq(False)
disagreements = exportable(pairs.loc[mask])
preview_columns = [c for c in keys + [
    'title_a', 'segment_text_a', 'frame_label_a', 'frame_label_b',
    'frame_type_a', 'frame_type_b', 'frame_explanation_a', 'frame_explanation_b',
    'types_only_a_with_polarity', 'types_only_b_with_polarity'
] if c in disagreements.columns]
display(disagreements[preview_columns].head(PREVIEW_ROWS))
print(f'{len(disagreements):,} disagreeing segments for {NAMES[INSPECT_METRIC]}.')

## 9. Save the report and provenance
The report records definitions and headline findings. `coverage.json` records input paths, alignment keys, coverage, and quality counts.

In [ ]:
coverage = {"model_a": names[0], "model_b": names[1], "input_a": str(CSV_A.resolve()),
            "input_b": str(CSV_B.resolve()), "join_keys": keys,
            "input_rows_a": len(a), "input_rows_b": len(b), "matched_rows": len(pairs),
            "only_a": int(joined._merge.eq("left_only").sum()),
            "only_b": int(joined._merge.eq("right_only").sum()),
            "matched_rows_with_quality_issues_or_warnings": int(pairs.quality_reasons.ne("").sum()),
            "quality_reason_counts": pairs.quality_reasons.str.split(";").explode().loc[lambda s: s.ne("")].value_counts().to_dict(),
            "grouping_columns_from_model_a": group_columns}
(out / "coverage.json").write_text(json.dumps(coverage, indent=2, ensure_ascii=False), encoding="utf-8")
text = [f"{names[0]} vs {names[1]}", f"Matched segments: {len(pairs):,}",
        f"Unmatched: A={coverage['only_a']:,}, B={coverage['only_b']:,}", ""]
for _, r in summary.iterrows():
    pct = f"{r.agreement_pct:.2f}%" if pd.notna(r.agreement_pct) else "n/a"
    kappa_text = f"{r.cohens_kappa:.4f}" if pd.notna(r.cohens_kappa) else "n/a"
    text.append(f"{NAMES[r.metric]}: {pct} ({r.n_agree:,}/{r.n_eligible:,}), "
                f"excluded={r.n_excluded:,}, Cohen's kappa={kappa_text}")
text.extend(["", "Type agreement is exact unordered-set equality. Case and whitespace are normalized.",
    "Empty NOT_FRAMED sets count as agreement when both models predict no frame.",
    "Processing errors, text mismatches, and missing/invalid predictions are excluded; see quality_issues.csv.",
    "Count warnings do not exclude otherwise usable predictions. Each metric states its denominator.",
    "Jaccard measures partial overlap. Per-frame binary agreement includes joint absences.",
    "Cohen's kappa corrects for marginal chance agreement; undefined values are blank in CSVs.",
    "Kappa for frame sets treats each complete set as a nominal category.",
    "Grouped comparisons use model A metadata, as recorded in coverage.json.",
    "These statistics measure model agreement, not accuracy against human annotations."])
report = "\n".join(text) + "\n"
(out / "report.txt").write_text(report, encoding="utf-8")

print(report)

## 10. Visualizations
All charts are saved as PNG files and displayed below. Frame prevalence charts are omitted if no usable type predictions exist.

In [ ]:
if MAKE_PLOTS:
    charts(summary, confusion, frame_stats, distribution, out, names)
    from PIL import Image
    for filename in [
        'agreement.png', 'frame_label_confusion.png', 'frame_label_distribution.png',
        'frame_prevalence_without_polarity.png', 'frame_prevalence_with_polarity.png'
    ]:
        path = out / filename
        if filename.startswith('frame_prevalence') and frame_stats.empty:
            continue
        if path.exists():
            print(filename)
            with Image.open(path) as image:
                display(image.copy())
else:
    print('Plot generation disabled in settings.')

## 11. Output files
Statistics, diagnostics, and segment exports use UTF-8 with a BOM for opening Chinese text in Excel. Files with the same names are overwritten; use a fresh output directory to avoid retaining disabled outputs from older runs.

In [ ]:
print(f'Results saved to: {out.resolve()}')
display(pd.DataFrame([
    {'file': p.name, 'size_kb': round(p.stat().st_size / 1024, 1)}
    for p in sorted(out.iterdir()) if p.is_file()
]))